# Prophet + Gradient Boost Pipeline — No Holt-Winters
**Author:** Chaitra R  June 2026



---
### Run modes
| Mode | Training | Test | Prophet yearly seasonality |
|---|---|---|---|
| `1year` | 2018 Jan–Sep | 2018 Oct–Dec | Disabled |
| `2year` | 2012 full + 2018 Jan–Sep | 2018 Oct–Dec | Enabled |

## Cell 1: Configuration

In [23]:
RUN_MODE = '2year'   # OPTIONS: '1year' or '2year'

from pathlib import Path
import os

BASE_DIR  = Path('.')                           # CorrectReleases/
DATA_DIR  = BASE_DIR / 'data'
RUN_DIR   = BASE_DIR / ('run1_1year' if RUN_MODE == '1year' else 'run2_2year')

# Reuse existing preprocessed/engineered data from main pipeline
PROCESSED = RUN_DIR / 'processed'
MODEL_DIR = RUN_DIR / 'models'
SELECTED  = BASE_DIR / 'selected_buildings.csv'

# New output folder — separate from main pipeline outputs
OUT_DIR   = BASE_DIR / f'run_prophet_gb_{RUN_MODE}'
PRED_DIR  = OUT_DIR / 'predictions'
RESULTS   = OUT_DIR / 'results'

for d in [PRED_DIR, RESULTS / 'plots', RESULTS / 'evaluation']:
    d.mkdir(parents=True, exist_ok=True)

TEST_CUTOFF = '2018-10-01'

print(f'RUN_MODE  : {RUN_MODE}')
print(f'Input     : {PROCESSED}')
print(f'Output    : {OUT_DIR}')
print(f'Models    : reusing {MODEL_DIR}/gb_model.pkl (already trained)')

RUN_MODE  : 2year
Input     : run2_2year/processed
Output    : run_prophet_gb_2year
Models    : reusing run2_2year/models/gb_model.pkl (already trained)


## Cell 2: Imports

In [24]:
import pandas as pd
import numpy as np
import pickle
import warnings
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_squared_log_error
)
from prophet import Prophet

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
print('✓ All imports successful')

✓ All imports successful


## Cell 3: Metrics Helpers

In [25]:
def compute_metrics(y_true, y_pred, label=''):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    clean  = y_true > 0
    yt, yp = y_true[clean], y_pred[clean].clip(0.001)
    mape_m = yt > 0.5
    return {
        'model' : label,
        'rmsle' : np.sqrt(mean_squared_log_error(yt, yp)),
        'rmse'  : np.sqrt(mean_squared_error(yt, yp)),
        'mae'   : mean_absolute_error(yt, yp),
        'r2'    : r2_score(yt, yp),
        'mape'  : np.mean(np.abs((yt[mape_m]-yp[mape_m])/yt[mape_m]))*100
                  if mape_m.sum() > 0 else np.nan,
    }

def daily_r2(df_in, pred_col):
    d = df_in.copy()
    d['date'] = pd.to_datetime(d['timestamp']).dt.date
    daily = d.groupby(['building_id','date']).agg(
        actual=('total_kwh', 'sum'),
        pred=(pred_col, 'sum')
    ).reset_index()
    clean = daily['actual'] > 0
    return r2_score(daily['actual'][clean], daily['pred'][clean])

def print_metrics(metrics_list):
    print(f'\n{"Model":<25} {"RMSLE":>8} {"RMSE":>8} {"MAE":>8} {"R²":>8} {"MAPE%":>8}')
    print('-'*68)
    for m in metrics_list:
        print(f'  {m["model"]:<23} {m["rmsle"]:>8.4f} {m["rmse"]:>8.4f} '
              f'{m["mae"]:>8.4f} {m["r2"]:>8.4f} {m["mape"]:>8.2f}')

print('✓ Metrics helpers ready')

✓ Metrics helpers ready


## Cell 4: Load Existing GB Predictions
GB is already trained — reuse predictions from main pipeline.  
No retraining needed.

In [26]:
# Reuse GB predictions from the existing run
gb_pred_path = RUN_DIR / 'predictions' / 'gb_predictions.parquet'
print(f'Loading GB predictions from: {gb_pred_path}')

gb_df = pd.read_parquet(gb_pred_path)
gb_df['building_id'] = gb_df['building_id'].astype(str)

tc = pd.Timestamp(TEST_CUTOFF)
gb_test = gb_df[gb_df['timestamp'] >= tc].copy()

print(f'✓ GB predictions loaded: {gb_test.shape}')
print(f'  Buildings : {gb_test["building_id"].nunique()}')
print(f'  Period    : {gb_test["timestamp"].min()} → {gb_test["timestamp"].max()}')
print(f'  Mean kWh  : {gb_test["total_kwh"].mean():.4f}')

m_gb = compute_metrics(gb_test['total_kwh'], gb_test['gb_prediction'], 'Gradient Boost')
print(f'\nGB standalone: RMSLE={m_gb["rmsle"]:.4f}  R²={m_gb["r2"]:.4f}  '
      f'MAE={m_gb["mae"]:.4f}  MAPE={m_gb["mape"]:.2f}%')

Loading GB predictions from: run2_2year/predictions/gb_predictions.parquet
✓ GB predictions loaded: (220900, 75)
  Buildings : 100
  Period    : 2018-10-01 00:00:00 → 2019-01-01 00:00:00
  Mean kWh  : 2.4464

GB standalone: RMSLE=0.1888  R²=0.8687  MAE=0.4779  MAPE=21.13%


## Cell 5: Prophet Model — Per Building

Same Prophet configuration as main pipeline.  
For `2year` mode, yearly seasonality is enabled — the key differentiator vs Holt-Winters.  
**No Holt-Winters smoothing applied before or after Prophet fitting.**

In [41]:
df = pd.read_parquet(PROCESSED / 'preprocessed_data.parquet')
df['cdh'] = np.maximum(0, df['temp_c'] - 18.0)
df['hdh'] = np.maximum(0, 18.0 - df['temp_c'])

if RUN_MODE == '1year':
    df_train = df[df['timestamp'] < tc].copy()
else:
    df_train = df[
        (df['data_year'] == 2012) |
        ((df['data_year'] == 2018) & (df['timestamp'] < tc))
    ].copy()
df_test = df[(df['data_year'] == 2018) & (df['timestamp'] >= tc)].copy()

print(f'Prophet training: {df_train.shape}  Test: {df_test.shape}')
print(f'Yearly seasonality: {"ENABLED" if RUN_MODE == "2year" else "DISABLED"}')
print(f'No Holt-Winters smoothing applied — raw time series fed directly to Prophet')
print()

predictions_list = []
failed = []
t0 = time.time()

for idx, bid in enumerate(df['building_id'].unique(), 1):
    b_train = df_train[df_train['building_id'] == bid].copy()
    b_test  = df_test[df_test['building_id'] == bid].copy()
    if len(b_train) < 100 or len(b_test) == 0:
        continue
    try:
        b_train['log_kwh'] = np.log1p(b_train['total_kwh'].clip(0))
        pt = b_train[['timestamp','log_kwh','temp_c','humidity_pct',
                      'ghi_wm2','cdh','hdh']].copy()
        pt.columns = ['ds','y','temp_c','humidity_pct','ghi_wm2','cdh','hdh']

        zone = b_train['climate_zone'].iloc[0]
        cdh_s = 2.0 if zone in ['2B','4B'] else 0.5 if zone == '5B' else 1.0
        hdh_s = 2.0 if zone == '5B' else 1.0

        # ── Prophet — NO Holt-Winters pre-smoothing ──────────
        m = Prophet(
            growth                 = 'linear',
            yearly_seasonality     = False,
            daily_seasonality      = False,
            weekly_seasonality     = True,
            seasonality_prior_scale= 0.1,
            changepoint_prior_scale= 0.005,
        )
        m.add_seasonality('daily',   period=1,     fourier_order=10, prior_scale=0.1)
        m.add_seasonality('weekly',  period=7,     fourier_order=5,  prior_scale=0.05)
        m.add_seasonality('monthly', period=30.5,  fourier_order=3,  prior_scale=0.05)
        if RUN_MODE == '2year':
            m.add_seasonality('yearly', period=365.25, fourier_order=5, prior_scale=0.1)

        for reg, ps in [('temp_c', 0.5), ('humidity_pct', 0.3),
                        ('ghi_wm2', 0.5), ('cdh', cdh_s), ('hdh', hdh_s)]:
            m.add_regressor(reg, prior_scale=ps, standardize=True)

        m.fit(pt)

        future = m.make_future_dataframe(
            periods=len(b_test), freq='h', include_history=False)
        future['temp_c']       = b_test['temp_c'].values
        future['humidity_pct'] = b_test['humidity_pct'].values
        future['ghi_wm2']      = b_test['ghi_wm2'].values
        future['cdh']          = b_test['cdh'].values
        future['hdh']          = b_test['hdh'].values

        fc = m.predict(future)
        b_test = b_test.copy()
        b_test['prophet_prediction'] = np.expm1(fc['yhat'].values).clip(0)
        b_test['prophet_trend']      = fc['trend'].values
        b_test['prophet_residual']   = b_test['total_kwh'] - b_test['prophet_prediction']
        predictions_list.append(b_test)

        if idx % 10 == 0:
            print(f'  [{idx:3d}/100] {bid}  zone={zone}  '
                  f'elapsed={time.time()-t0:.0f}s')
    except Exception as e:
        failed.append({'bid': bid, 'error': str(e)[:80]})

prophet_df = pd.concat(predictions_list, ignore_index=True)
prophet_df['building_id'] = prophet_df['building_id'].astype(str)
prophet_df.to_parquet(PRED_DIR / 'prophet_predictions.parquet', index=False)

m_p = compute_metrics(prophet_df['total_kwh'], prophet_df['prophet_prediction'], 'Prophet')
print(f'\n✓ Prophet done in {time.time()-t0:.0f}s  |  {len(failed)} failed')
print(f'Prophet standalone: RMSLE={m_p["rmsle"]:.4f}  R²={m_p["r2"]:.4f}  '
      f'MAE={m_p["mae"]:.4f}')

Prophet training: (1533500, 19)  Test: (220900, 19)
Yearly seasonality: ENABLED
No Holt-Winters smoothing applied — raw time series fed directly to Prophet



18:06:05 - cmdstanpy - INFO - Chain [1] start processing
18:06:06 - cmdstanpy - INFO - Chain [1] done processing
18:06:07 - cmdstanpy - INFO - Chain [1] start processing
18:06:07 - cmdstanpy - INFO - Chain [1] done processing
18:06:08 - cmdstanpy - INFO - Chain [1] start processing
18:06:09 - cmdstanpy - INFO - Chain [1] done processing
18:06:10 - cmdstanpy - INFO - Chain [1] start processing
18:06:10 - cmdstanpy - INFO - Chain [1] done processing
18:06:11 - cmdstanpy - INFO - Chain [1] start processing
18:06:12 - cmdstanpy - INFO - Chain [1] done processing
18:06:13 - cmdstanpy - INFO - Chain [1] start processing
18:06:14 - cmdstanpy - INFO - Chain [1] done processing
18:06:15 - cmdstanpy - INFO - Chain [1] start processing
18:06:15 - cmdstanpy - INFO - Chain [1] done processing
18:06:16 - cmdstanpy - INFO - Chain [1] start processing
18:06:17 - cmdstanpy - INFO - Chain [1] done processing
18:06:18 - cmdstanpy - INFO - Chain [1] start processing
18:06:18 - cmdstanpy - INFO - Chain [1]

  [ 10/100] 158867  zone=3C  elapsed=16s


18:06:21 - cmdstanpy - INFO - Chain [1] start processing
18:06:21 - cmdstanpy - INFO - Chain [1] done processing
18:06:22 - cmdstanpy - INFO - Chain [1] start processing
18:06:23 - cmdstanpy - INFO - Chain [1] done processing
18:06:24 - cmdstanpy - INFO - Chain [1] start processing
18:06:24 - cmdstanpy - INFO - Chain [1] done processing
18:06:25 - cmdstanpy - INFO - Chain [1] start processing
18:06:26 - cmdstanpy - INFO - Chain [1] done processing
18:06:27 - cmdstanpy - INFO - Chain [1] start processing
18:06:28 - cmdstanpy - INFO - Chain [1] done processing
18:06:29 - cmdstanpy - INFO - Chain [1] start processing
18:06:29 - cmdstanpy - INFO - Chain [1] done processing
18:06:30 - cmdstanpy - INFO - Chain [1] start processing
18:06:31 - cmdstanpy - INFO - Chain [1] done processing
18:06:32 - cmdstanpy - INFO - Chain [1] start processing
18:06:32 - cmdstanpy - INFO - Chain [1] done processing
18:06:33 - cmdstanpy - INFO - Chain [1] start processing
18:06:34 - cmdstanpy - INFO - Chain [1]

  [ 20/100] 218960  zone=3B  elapsed=31s


...

  [ 80/100] 498749  zone=4C  elapsed=132s


18:08:17 - cmdstanpy - INFO - Chain [1] start processing
18:08:18 - cmdstanpy - INFO - Chain [1] done processing
18:08:19 - cmdstanpy - INFO - Chain [1] start processing
18:08:19 - cmdstanpy - INFO - Chain [1] done processing
18:08:20 - cmdstanpy - INFO - Chain [1] start processing
18:08:21 - cmdstanpy - INFO - Chain [1] done processing
18:08:22 - cmdstanpy - INFO - Chain [1] start processing
18:08:22 - cmdstanpy - INFO - Chain [1] done processing
18:08:23 - cmdstanpy - INFO - Chain [1] start processing
18:08:24 - cmdstanpy - INFO - Chain [1] done processing
18:08:25 - cmdstanpy - INFO - Chain [1] start processing
18:08:25 - cmdstanpy - INFO - Chain [1] done processing
18:08:26 - cmdstanpy - INFO - Chain [1] start processing
18:08:27 - cmdstanpy - INFO - Chain [1] done processing
18:08:28 - cmdstanpy - INFO - Chain [1] start processing
18:08:28 - cmdstanpy - INFO - Chain [1] done processing
18:08:29 - cmdstanpy - INFO - Chain [1] start processing
18:08:30 - cmdstanpy - INFO - Chain [1]

  [ 90/100] 544145  zone=4B  elapsed=148s


18:08:33 - cmdstanpy - INFO - Chain [1] start processing
18:08:33 - cmdstanpy - INFO - Chain [1] done processing
18:08:34 - cmdstanpy - INFO - Chain [1] start processing
18:08:35 - cmdstanpy - INFO - Chain [1] done processing
18:08:36 - cmdstanpy - INFO - Chain [1] start processing
18:08:36 - cmdstanpy - INFO - Chain [1] done processing
18:08:37 - cmdstanpy - INFO - Chain [1] start processing
18:08:38 - cmdstanpy - INFO - Chain [1] done processing
18:08:39 - cmdstanpy - INFO - Chain [1] start processing
18:08:40 - cmdstanpy - INFO - Chain [1] done processing
18:08:41 - cmdstanpy - INFO - Chain [1] start processing
18:08:42 - cmdstanpy - INFO - Chain [1] done processing
18:08:43 - cmdstanpy - INFO - Chain [1] start processing
18:08:43 - cmdstanpy - INFO - Chain [1] done processing
18:08:44 - cmdstanpy - INFO - Chain [1] start processing
18:08:45 - cmdstanpy - INFO - Chain [1] done processing
18:08:46 - cmdstanpy - INFO - Chain [1] start processing
18:08:46 - cmdstanpy - INFO - Chain [1]

  [100/100] 9063  zone=4B  elapsed=164s

✓ Prophet done in 164s  |  0 failed
Prophet standalone: RMSLE=0.2703  R²=0.6111  MAE=0.7324


## Cell 6: Prophet + GB Ensemble — Zone-Aware Weights

Grid search over Prophet weights 0.0–0.50 per climate zone.  
No Holt-Winters included — the ensemble is purely Prophet + GB.

In [42]:
prophet_df = pd.read_parquet(PRED_DIR / 'prophet_predictions.parquet')
prophet_df['building_id'] = prophet_df['building_id'].astype(str)

# Merge GB + Prophet on building_id + timestamp
df = gb_test[['building_id','timestamp','total_kwh',
              'climate_zone','gb_prediction']].merge(
    prophet_df[['building_id','timestamp','prophet_prediction']],
    on=['building_id','timestamp'], how='inner'
)
print(f'Merged: {df.shape}  |  Buildings: {df["building_id"].nunique()}')

clean_df = df[df['total_kwh'] > 0].copy()
best_weights = {}

print('\nOptimising zone weights (Prophet + GB only):')
print(f'{"Zone":<6} {"Prophet W":>10} {"GB W":>8} {"RMSLE":>8}')
print('-'*38)

for zone in sorted(df['climate_zone'].dropna().unique()):
    z = clean_df[clean_df['climate_zone'] == zone]
    best_r, best_w = 999, 0.0
    for w_p in np.arange(0.0, 0.51, 0.025):
        pred = (
            w_p * z['prophet_prediction'] +
            (1 - w_p) * z['gb_prediction']
        ).clip(0.001)
        r = np.sqrt(mean_squared_log_error(z['total_kwh'], pred))
        if r < best_r:
            best_r, best_w = r, w_p
    best_weights[zone] = {'prophet': best_w, 'gb': round(1 - best_w, 3)}
    print(f'  {zone:<4}  {best_w:>10.3f}  {1-best_w:>8.3f}  {best_r:>8.4f}')

# Apply weights
df['w_p']  = df['climate_zone'].map(
    {z: v['prophet'] for z, v in best_weights.items()})
df['w_gb'] = df['climate_zone'].map(
    {z: v['gb'] for z, v in best_weights.items()})
df['ensemble_prediction'] = (
    df['w_p'] * df['prophet_prediction'] +
    df['w_gb'] * df['gb_prediction']
).clip(0)

df.to_parquet(PRED_DIR / 'ensemble_predictions.parquet', index=False)

# Save weights
import json
with open(RESULTS / 'ensemble_weights.json', 'w') as f:
    json.dump(best_weights, f, indent=2)

print(f'\n✓ Ensemble predictions saved')

Merged: (220900, 6)  |  Buildings: 100

Optimising zone weights (Prophet + GB only):
Zone    Prophet W     GB W    RMSLE
--------------------------------------
  2B         0.150     0.850    0.1611
  3B         0.125     0.875    0.1735
  3C         0.200     0.800    0.2001
  4B         0.125     0.875    0.2008
  4C         0.225     0.775    0.1759
  5B         0.175     0.825    0.1859

✓ Ensemble predictions saved


## Cell 7: Full Metrics Comparison

Compares Prophet-only, GB-only, and Prophet+GB ensemble.  
No Holt-Winters row — it has been dropped from this pipeline entirely.

In [43]:
df = pd.read_parquet(PRED_DIR / 'ensemble_predictions.parquet')

m_gb  = compute_metrics(df['total_kwh'], df['gb_prediction'],       'Gradient Boost')
m_p   = compute_metrics(df['total_kwh'], df['prophet_prediction'],  'Prophet')
m_ens = compute_metrics(df['total_kwh'], df['ensemble_prediction'], 'Prophet + GB Ensemble')

# Daily R²
dr2_gb  = daily_r2(df, 'gb_prediction')
dr2_p   = daily_r2(df, 'prophet_prediction')
dr2_ens = daily_r2(df, 'ensemble_prediction')

print('='*68)
print('PROPHET + GB ENSEMBLE — NO HOLT-WINTERS')
print('='*68)
print_metrics([m_gb, m_p, m_ens])

print(f'\nDaily R²:')
print(f'  GB standalone    : {dr2_gb:.4f}')
print(f'  Prophet standalone: {dr2_p:.4f}')
print(f'  Ensemble         : {dr2_ens:.4f}')

# Compare vs original hybrid (which included Holt-Winters in reporting)
print(f'\nComparison vs original ensemble (from main pipeline):')
print(f'  Original ensemble RMSLE (with HW in report) : 0.1879')
print(f'  This ensemble RMSLE (Prophet + GB only)      : {m_ens["rmsle"]:.4f}')
delta = m_ens['rmsle'] - 0.1879
print(f'  Delta                                        : {delta:+.4f}  '
      f'({"✓ same or better" if delta <= 0.0001 else "~ marginal difference"})')

# Prophet contribution in this ensemble
rmsle_delta = m_gb['rmsle'] - m_ens['rmsle']
print(f'\nProphet contribution (GB-only → Ensemble):')
print(f'  RMSLE: {m_gb["rmsle"]:.4f} → {m_ens["rmsle"]:.4f}  '
      f'(delta: {rmsle_delta:+.4f} / {rmsle_delta/m_gb["rmsle"]*100:+.2f}%)')

# Save all metrics
import pandas as pd
metrics_df = pd.DataFrame([m_gb, m_p, m_ens])
metrics_df['daily_r2'] = [dr2_gb, dr2_p, dr2_ens]
metrics_df.to_csv(RESULTS / 'evaluation' / 'final_metrics.csv', index=False)
print(f'\n✓ Metrics saved: {RESULTS}/evaluation/final_metrics.csv')

PROPHET + GB ENSEMBLE — NO HOLT-WINTERS

Model                        RMSLE     RMSE      MAE       R²    MAPE%
--------------------------------------------------------------------
  Gradient Boost            0.1888   0.9811   0.4779   0.8687    21.13
  Prophet                   0.2703   1.6883   0.7324   0.6111    28.75
  Prophet + GB Ensemble     0.1856   1.0008   0.4790   0.8633    20.81

Daily R²:
  GB standalone    : 0.9925
  Prophet standalone: 0.8767
  Ensemble         : 0.9893

Comparison vs original ensemble (from main pipeline):
  Original ensemble RMSLE (with HW in report) : 0.1879
  This ensemble RMSLE (Prophet + GB only)      : 0.1856
  Delta                                        : -0.0023  (✓ same or better)

Prophet contribution (GB-only → Ensemble):
  RMSLE: 0.1888 → 0.1856  (delta: +0.0032 / +1.70%)

✓ Metrics saved: run_prophet_gb_2year/results/evaluation/final_metrics.csv


## Cell 8: Visualisation — Model Comparison

In [44]:
df = pd.read_parquet(PRED_DIR / 'ensemble_predictions.parquet')
df['date'] = pd.to_datetime(df['timestamp']).dt.date

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models = [
    ('gb_prediction',       'Gradient Boost', '#1D9E75'),
    ('prophet_prediction',  'Prophet',        '#E07B39'),
    ('ensemble_prediction', 'Ensemble',       '#1B4F72'),
]

# Compute metrics per model for title
m_list = {
    'gb_prediction'      : compute_metrics(df['total_kwh'], df['gb_prediction']),
    'prophet_prediction' : compute_metrics(df['total_kwh'], df['prophet_prediction']),
    'ensemble_prediction': compute_metrics(df['total_kwh'], df['ensemble_prediction']),
}

for ax, (col, name, color) in zip(axes, models):
    clean = df[df['total_kwh'] > 0]
    sample = clean.sample(min(3000, len(clean)), random_state=42)
    ax.scatter(sample['total_kwh'], sample[col],
               alpha=0.3, s=8, color=color)
    lim = max(sample['total_kwh'].max(), sample[col].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'gray', linewidth=1, linestyle='--')
    m = m_list[col]
    ax.set_title(
        f'{name}\nRMSLE={m["rmsle"]:.4f}  R²={m["r2"]:.4f}',
        fontweight='bold', fontsize=10
    )
    ax.set_xlabel('Actual kWh')
    ax.set_ylabel('Predicted kWh')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(
    f'Prophet + GB Ensemble — No Holt-Winters\n'
    f'Test: Oct–Dec 2018 | {RUN_MODE} training | 100 buildings',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'model_comparison_scatter.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('✓ Scatter plot saved')

# ── Daily aggregate time series ───────────────────────────────
daily = df.groupby('date').agg(
    actual=('total_kwh', 'sum'),
    gb=('gb_prediction', 'sum'),
    prophet=('prophet_prediction', 'sum'),
    ensemble=('ensemble_prediction', 'sum')
).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily['date'], daily['actual'],   color='black',   lw=1.5, label='Actual', zorder=3)
ax.plot(daily['date'], daily['ensemble'], color='#1B4F72', lw=1.5, label='Ensemble', zorder=2)
ax.plot(daily['date'], daily['gb'],       color='#1D9E75', lw=1,   label='GB only',  alpha=0.7)
ax.plot(daily['date'], daily['prophet'],  color='#E07B39', lw=1,   label='Prophet only', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Daily kWh (all buildings)')
ax.set_title('Daily Aggregate: Prophet + GB vs Standalone Models', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'daily_aggregate.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ Daily aggregate plot saved')

✓ Scatter plot saved
✓ Daily aggregate plot saved


## Cell 9: Per-Zone Analysis

In [32]:
df = pd.read_parquet(PRED_DIR / 'ensemble_predictions.parquet')

print(f'Per-zone results — Prophet + GB Ensemble (no Holt-Winters):')
print(f'{"Zone":<6} {"Prophet W":>10} {"GB RMSLE":>10} '
      f'{"Ensemble RMSLE":>15} {"R²":>8} {"Prophet gain":>13}')
print('-'*68)

zone_results = []
for zone in sorted(df['climate_zone'].dropna().unique()):
    z      = df[df['climate_zone'] == zone]
    w_p    = z['w_p'].iloc[0] if 'w_p' in z.columns else best_weights.get(zone, {}).get('prophet', 0)
    m_z_gb = compute_metrics(z['total_kwh'], z['gb_prediction'])
    m_z_en = compute_metrics(z['total_kwh'], z['ensemble_prediction'])
    gain   = m_z_gb['rmsle'] - m_z_en['rmsle']
    zone_results.append({
        'zone': zone, 'prophet_weight': w_p,
        'gb_rmsle': m_z_gb['rmsle'],
        'ensemble_rmsle': m_z_en['rmsle'],
        'r2': m_z_en['r2'],
        'prophet_gain': gain
    })
    print(f'  {zone:<4}  {w_p:>10.3f}  {m_z_gb["rmsle"]:>10.4f}  '
          f'{m_z_en["rmsle"]:>15.4f}  {m_z_en["r2"]:>8.4f}  '
          f'{gain:>+13.4f}')

zone_df = pd.DataFrame(zone_results)
zone_df.to_csv(RESULTS / 'evaluation' / 'per_zone_metrics.csv', index=False)
print(f'\n✓ Per-zone results saved')

Per-zone results — Prophet + GB Ensemble (no Holt-Winters):
Zone    Prophet W   GB RMSLE  Ensemble RMSLE       R²  Prophet gain
--------------------------------------------------------------------
  2B         0.150      0.1649           0.1611    0.7513        +0.0038
  3B         0.125      0.1756           0.1735    0.9147        +0.0021
  3C         0.200      0.2047           0.2001    0.7663        +0.0045
  4B         0.125      0.2030           0.2008    0.8909        +0.0022
  4C         0.225      0.1801           0.1759    0.7761        +0.0042
  5B         0.175      0.1895           0.1859    0.8082        +0.0036

✓ Per-zone results saved
